# Спринт ревью №3
## Тема: «Полиморфизм и Generic классы»

**Язык:** C# (Polyglot Notebook / .NET Interactive)

**Цель работы:** продемонстрировать полиморфизм, перегрузку и перекрытие методов, а также обобщённые типы (Generic) на примере класса «Компьютер» и его компонентов.

**Структура программы:**

| Элемент | Назначение |
|---|---|
| `Component` (абстрактный) | общие свойства и методы всех компонентов |
| `IInstallable` | интерфейс с методом `Install()` |
| `CPU`, `RAM`, `GPU`, `HDD` | конкретные компоненты, наследуют `Component` и реализуют `IInstallable` |
| `Computer` | хранит список компонентов, generic-методы `AddComponent<T>`, `RemoveComponent<T>`, `GetTotalPrice()` |
| `ComputerFactory` (статический) | создание готовых конфигураций компьютеров |

> Ячейки нужно запускать строго сверху вниз.

## 1. Абстрактный класс `Component`

- Содержит общие свойства: `Name`, `Manufacturer`, `Price`.
- `GetInfo()` - **виртуальный** метод, наследники перекрывают его (`override`).
- `ToString()` переопределён и возвращает результат `GetInfo()`.
- Объект абстрактного класса создать нельзя (`new Component(...)` вызовет ошибку).

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

public abstract class Component
{
    public string Name { get; set; }          // название модели
    public string Manufacturer { get; set; }  // производитель
    public decimal Price { get; set; }        // цена

    protected Component(string name, string manufacturer, decimal price)
    {
        Name = name;
        Manufacturer = manufacturer;
        Price = price;
    }

    // virtual - наследники могут перекрыть метод
    public virtual string GetInfo()
    {
        return $"{Name} ({Manufacturer}), цена: {Price:F2} руб.";
    }

    // Переопределение ToString() для удобного представления компонента
    public override string ToString()
    {
        return GetInfo();
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

## 2. Интерфейс `IInstallable`

Определяет метод установки компонента в компьютер. Интерфейс объявляется до конкретных классов, которые его реализуют.

In [2]:
public interface IInstallable
{
    string Install();   // возвращает сообщение об установке
}

## 3. Класс `CPU`

- Наследует `Component` и реализует `IInstallable`.
- Перекрывает `GetInfo()` (добавляет число ядер и частоту).
- Реализует `Install()`.
- Перегружает операторы `==` и `!=`. При перегрузке этих операторов также переопределяются `Equals()` и `GetHashCode()` - так требует C#.
- Процессоры считаются равными, если совпадают название, производитель, число ядер и частота (цена не учитывается).

In [3]:
public class CPU : Component, IInstallable
{
    public int Cores { get; set; }
    public double FrequencyGHz { get; set; }

    public CPU(string name, string manufacturer, decimal price, int cores, double frequencyGHz)
        : base(name, manufacturer, price)
    {
        Cores = cores;
        FrequencyGHz = frequencyGHz;
    }

    public override string GetInfo()
    {
        return $"CPU: {base.GetInfo()}, ядер: {Cores}, частота: {FrequencyGHz:F1} ГГц";
    }

    public string Install()
    {
        return $"Процессор {Name} установлен в сокет материнской платы.";
    }

    public static bool operator ==(CPU left, CPU right)
    {
        if (ReferenceEquals(left, right)) return true;
        if (left is null || right is null) return false;
        return left.Name == right.Name
            && left.Manufacturer == right.Manufacturer
            && left.Cores == right.Cores
            && left.FrequencyGHz == right.FrequencyGHz;
    }

    public static bool operator !=(CPU left, CPU right)
    {
        return !(left == right);
    }

    public override bool Equals(object obj)
    {
        return obj is CPU other && this == other;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(Name, Manufacturer, Cores, FrequencyGHz);
    }
}

## 4. Класс `RAM`

Оперативная память: объём и тип (DDR4, DDR5). Устроена так же, как `CPU`: перекрытие `GetInfo()`, метод `Install()`, операторы `==` и `!=`.

In [4]:
public class RAM : Component, IInstallable
{
    public int CapacityGB { get; set; }
    public string MemoryType { get; set; }

    public RAM(string name, string manufacturer, decimal price, int capacityGB, string memoryType)
        : base(name, manufacturer, price)
    {
        CapacityGB = capacityGB;
        MemoryType = memoryType;
    }

    public override string GetInfo()
    {
        return $"RAM: {base.GetInfo()}, объём: {CapacityGB} ГБ, тип: {MemoryType}";
    }

    public string Install()
    {
        return $"Модуль памяти {Name} установлен в слот DIMM.";
    }

    public static bool operator ==(RAM left, RAM right)
    {
        if (ReferenceEquals(left, right)) return true;
        if (left is null || right is null) return false;
        return left.Name == right.Name
            && left.Manufacturer == right.Manufacturer
            && left.CapacityGB == right.CapacityGB
            && left.MemoryType == right.MemoryType;
    }

    public static bool operator !=(RAM left, RAM right)
    {
        return !(left == right);
    }

    public override bool Equals(object obj)
    {
        return obj is RAM other && this == other;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(Name, Manufacturer, CapacityGB, MemoryType);
    }
}

## 5. Класс `GPU`

Видеокарта: объём видеопамяти.

In [5]:
public class GPU : Component, IInstallable
{
    public int VideoMemoryGB { get; set; }

    public GPU(string name, string manufacturer, decimal price, int videoMemoryGB)
        : base(name, manufacturer, price)
    {
        VideoMemoryGB = videoMemoryGB;
    }

    public override string GetInfo()
    {
        return $"GPU: {base.GetInfo()}, видеопамять: {VideoMemoryGB} ГБ";
    }

    public string Install()
    {
        return $"Видеокарта {Name} установлена в слот PCI-Express.";
    }

    public static bool operator ==(GPU left, GPU right)
    {
        if (ReferenceEquals(left, right)) return true;
        if (left is null || right is null) return false;
        return left.Name == right.Name
            && left.Manufacturer == right.Manufacturer
            && left.VideoMemoryGB == right.VideoMemoryGB;
    }

    public static bool operator !=(GPU left, GPU right)
    {
        return !(left == right);
    }

    public override bool Equals(object obj)
    {
        return obj is GPU other && this == other;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(Name, Manufacturer, VideoMemoryGB);
    }
}

## 6. Класс `HDD`

Жёсткий диск: объём и скорость вращения шпинделя.

In [6]:
public class HDD : Component, IInstallable
{
    public int CapacityGB { get; set; }
    public int Rpm { get; set; }

    public HDD(string name, string manufacturer, decimal price, int capacityGB, int rpm)
        : base(name, manufacturer, price)
    {
        CapacityGB = capacityGB;
        Rpm = rpm;
    }

    public override string GetInfo()
    {
        return $"HDD: {base.GetInfo()}, объём: {CapacityGB} ГБ, скорость: {Rpm} об/мин";
    }

    public string Install()
    {
        return $"Жёсткий диск {Name} подключён к интерфейсу SATA.";
    }

    public static bool operator ==(HDD left, HDD right)
    {
        if (ReferenceEquals(left, right)) return true;
        if (left is null || right is null) return false;
        return left.Name == right.Name
            && left.Manufacturer == right.Manufacturer
            && left.CapacityGB == right.CapacityGB
            && left.Rpm == right.Rpm;
    }

    public static bool operator !=(HDD left, HDD right)
    {
        return !(left == right);
    }

    public override bool Equals(object obj)
    {
        return obj is HDD other && this == other;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(Name, Manufacturer, CapacityGB, Rpm);
    }
}

## 7. Класс `Computer`

- Хранит список `List<Component>` (полиморфизм: в нём лежат объекты разных классов).
- `AddComponent<T>(T component) where T : Component, IInstallable` - добавляет только такие компоненты, которые можно установить.
- `RemoveComponent<T>(T component) where T : Component` - удаляет компонент (сравнение через переопределённый `Equals`).
- `GetTotalPrice()` - общая стоимость. Есть **перегрузка** `GetTotalPrice(decimal discountPercent)` - со скидкой.
- `GetComponents<T>()` - generic-выборка компонентов определённого типа.

In [7]:
public class Computer
{
    private readonly List<Component> _components = new List<Component>();

    public string Name { get; set; }

    public IReadOnlyList<Component> Components => _components;

    public Computer(string name)
    {
        Name = name;
    }

    // Generic-метод с ограничениями: T должен быть Component и реализовывать IInstallable
    public void AddComponent<T>(T component) where T : Component, IInstallable
    {
        if (component == null)
            throw new ArgumentNullException(nameof(component));

        Console.WriteLine($"[{Name}] {component.Install()}");
        _components.Add(component);
    }

    // Generic-метод удаления; возвращает true, если компонент был найден
    public bool RemoveComponent<T>(T component) where T : Component
    {
        bool removed = _components.Remove(component);
        Console.WriteLine(removed
            ? $"[{Name}] Компонент удалён: {component.Name}"
            : $"[{Name}] Компонент не найден: {component.Name}");
        return removed;
    }

    // Выборка компонентов нужного типа
    public List<T> GetComponents<T>() where T : Component
    {
        return _components.OfType<T>().ToList();
    }

    // Общая стоимость
    public decimal GetTotalPrice()
    {
        return _components.Sum(c => c.Price);
    }

    // Перегрузка метода: стоимость со скидкой в процентах
    public decimal GetTotalPrice(decimal discountPercent)
    {
        return GetTotalPrice() * (100m - discountPercent) / 100m;
    }

    public void ShowConfiguration()
    {
        Console.WriteLine($"Конфигурация компьютера \"{Name}\":");
        foreach (Component c in _components)
        {
            Console.WriteLine("  - " + c.GetInfo());   // полиморфный вызов
        }
        Console.WriteLine($"Итого: {GetTotalPrice():F2} руб.");
    }
}

## 8. Дополнительное задание: статический класс `ComputerFactory`

Содержит методы для создания готовых конфигураций. Метод `CreateCustomComputer<TCpu, TRam, TGpu, THdd>` - **generic**: принимает компоненты конкретных типов и собирает из них компьютер.

In [8]:
public static class ComputerFactory
{
    // Готовая офисная конфигурация
    public static Computer CreateOfficeComputer()
    {
        var pc = new Computer("Офисный ПК");
        pc.AddComponent(new CPU("Core i3-12100", "Intel", 11000m, 4, 3.3));
        pc.AddComponent(new RAM("Vengeance 8GB", "Corsair", 2500m, 8, "DDR4"));
        pc.AddComponent(new HDD("Barracuda 1TB", "Seagate", 3500m, 1000, 7200));
        return pc;
    }

    // Готовая игровая конфигурация
    public static Computer CreateGamingComputer()
    {
        var pc = new Computer("Игровой ПК");
        pc.AddComponent(new CPU("Ryzen 7 7800X3D", "AMD", 38000m, 8, 4.2));
        pc.AddComponent(new RAM("Fury Beast 32GB", "Kingston", 9000m, 32, "DDR5"));
        pc.AddComponent(new GPU("GeForce RTX 4070", "NVIDIA", 60000m, 12));
        pc.AddComponent(new HDD("Barracuda 2TB", "Seagate", 5000m, 2000, 7200));
        return pc;
    }

    // Generic-метод: компьютер из любых компонентов нужных типов
    public static Computer CreateCustomComputer<TCpu, TRam, TGpu, THdd>(
        string name, TCpu cpu, TRam ram, TGpu gpu, THdd hdd)
        where TCpu : CPU
        where TRam : RAM
        where TGpu : GPU
        where THdd : HDD
    {
        var pc = new Computer(name);
        pc.AddComponent(cpu);
        pc.AddComponent(ram);
        pc.AddComponent(gpu);
        pc.AddComponent(hdd);
        return pc;
    }
}

## 9. Демонстрация работы программы

### 9.1. Создание компонентов, `GetInfo()` и `ToString()`

In [9]:
var cpuA = new CPU("Core i7-13700K", "Intel", 32000m, 16, 3.4);
var ramA = new RAM("Fury Beast 16GB", "Kingston", 5000m, 16, "DDR5");
var gpuA = new GPU("Radeon RX 7800 XT", "AMD", 48000m, 16);
var hddA = new HDD("Blue 1TB", "Western Digital", 3000m, 1000, 7200);

Console.WriteLine("GetInfo():");
Console.WriteLine(cpuA.GetInfo());
Console.WriteLine(ramA.GetInfo());
Console.WriteLine(gpuA.GetInfo());
Console.WriteLine(hddA.GetInfo());

Console.WriteLine();
Console.WriteLine("ToString():");
Console.WriteLine(cpuA.ToString());
Console.WriteLine(gpuA);   // Console.WriteLine сам вызывает ToString()

GetInfo():
CPU: Core i7-13700K (Intel), цена: 32000,00 руб., ядер: 16, частота: 3,4 ГГц
RAM: Fury Beast 16GB (Kingston), цена: 5000,00 руб., объём: 16 ГБ, тип: DDR5
GPU: Radeon RX 7800 XT (AMD), цена: 48000,00 руб., видеопамять: 16 ГБ
HDD: Blue 1TB (Western Digital), цена: 3000,00 руб., объём: 1000 ГБ, скорость: 7200 об/мин

ToString():
CPU: Core i7-13700K (Intel), цена: 32000,00 руб., ядер: 16, частота: 3,4 ГГц
GPU: Radeon RX 7800 XT (AMD), цена: 48000,00 руб., видеопамять: 16 ГБ


### 9.2. Операторы `==` и `!=`

`cpuB` имеет те же характеристики, что `cpuA`, но другую цену и является другим объектом - по нашему оператору они равны.

In [10]:
var cpuB = new CPU("Core i7-13700K", "Intel", 31000m, 16, 3.4);
var cpuC = new CPU("Ryzen 5 7600", "AMD", 21000m, 6, 3.8);
var ramB = new RAM("Fury Beast 16GB", "Kingston", 5200m, 16, "DDR5");

Console.WriteLine($"cpuA == cpuB: {cpuA == cpuB}");
Console.WriteLine($"cpuA != cpuB: {cpuA != cpuB}");
Console.WriteLine($"cpuA == cpuC: {cpuA == cpuC}");
Console.WriteLine($"cpuA != cpuC: {cpuA != cpuC}");
Console.WriteLine($"ramA == ramB: {ramA == ramB}");

cpuA == cpuB: True
cpuA != cpuB: False
cpuA == cpuC: False
cpuA != cpuC: True
ramA == ramB: True


### 9.3. Сборка компьютера: `AddComponent<T>`, `GetTotalPrice()`, перегрузка

In [11]:
var workPc = new Computer("Рабочий ПК");

workPc.AddComponent(cpuA);
workPc.AddComponent(ramA);
workPc.AddComponent(gpuA);
workPc.AddComponent(hddA);

Console.WriteLine();
workPc.ShowConfiguration();

Console.WriteLine();
Console.WriteLine($"GetTotalPrice(): {workPc.GetTotalPrice():F2} руб.");
Console.WriteLine($"GetTotalPrice(10) - со скидкой 10%: {workPc.GetTotalPrice(10m):F2} руб.");

var allGpus = workPc.GetComponents<GPU>();
Console.WriteLine($"Видеокарт в компьютере: {allGpus.Count}");

[Рабочий ПК] Процессор Core i7-13700K установлен в сокет материнской платы.
[Рабочий ПК] Модуль памяти Fury Beast 16GB установлен в слот DIMM.
[Рабочий ПК] Видеокарта Radeon RX 7800 XT установлена в слот PCI-Express.
[Рабочий ПК] Жёсткий диск Blue 1TB подключён к интерфейсу SATA.

Конфигурация компьютера "Рабочий ПК":
  - CPU: Core i7-13700K (Intel), цена: 32000,00 руб., ядер: 16, частота: 3,4 ГГц
  - RAM: Fury Beast 16GB (Kingston), цена: 5000,00 руб., объём: 16 ГБ, тип: DDR5
  - GPU: Radeon RX 7800 XT (AMD), цена: 48000,00 руб., видеопамять: 16 ГБ
  - HDD: Blue 1TB (Western Digital), цена: 3000,00 руб., объём: 1000 ГБ, скорость: 7200 об/мин
Итого: 88000,00 руб.

GetTotalPrice(): 88000,00 руб.
GetTotalPrice(10) - со скидкой 10%: 79200,00 руб.
Видеокарт в компьютере: 1


### 9.4. Полиморфизм: список `Component` и интерфейс `IInstallable`

In [12]:
Console.WriteLine("Полиморфный вызов GetInfo() через тип Component:");
foreach (Component part in workPc.Components)
{
    Console.WriteLine($"{part.GetType().Name,-4} -> {part.GetInfo()}");
}

Console.WriteLine();
Console.WriteLine("Вызов Install() через интерфейс IInstallable:");
foreach (IInstallable item in workPc.Components.OfType<IInstallable>())
{
    Console.WriteLine(item.Install());
}

Полиморфный вызов GetInfo() через тип Component:
CPU  -> CPU: Core i7-13700K (Intel), цена: 32000,00 руб., ядер: 16, частота: 3,4 ГГц
RAM  -> RAM: Fury Beast 16GB (Kingston), цена: 5000,00 руб., объём: 16 ГБ, тип: DDR5
GPU  -> GPU: Radeon RX 7800 XT (AMD), цена: 48000,00 руб., видеопамять: 16 ГБ
HDD  -> HDD: Blue 1TB (Western Digital), цена: 3000,00 руб., объём: 1000 ГБ, скорость: 7200 об/мин

Вызов Install() через интерфейс IInstallable:
Процессор Core i7-13700K установлен в сокет материнской платы.
Модуль памяти Fury Beast 16GB установлен в слот DIMM.
Видеокарта Radeon RX 7800 XT установлена в слот PCI-Express.
Жёсткий диск Blue 1TB подключён к интерфейсу SATA.


### 9.5. Удаление компонентов: `RemoveComponent<T>`

Удаление работает и для нового объекта `cpuB`, потому что он равен `cpuA` по переопределённому `Equals`. Попытка удалить отсутствующий компонент (`cpuC`) возвращает `false`.

In [13]:
workPc.RemoveComponent(hddA);   // удаляем жёсткий диск
workPc.RemoveComponent(cpuC);   // такого процессора нет
workPc.RemoveComponent(cpuB);   // равен cpuA, поэтому будет удалён

Console.WriteLine();
workPc.ShowConfiguration();

[Рабочий ПК] Компонент удалён: Blue 1TB
[Рабочий ПК] Компонент не найден: Ryzen 5 7600
[Рабочий ПК] Компонент удалён: Core i7-13700K

Конфигурация компьютера "Рабочий ПК":
  - RAM: Fury Beast 16GB (Kingston), цена: 5000,00 руб., объём: 16 ГБ, тип: DDR5
  - GPU: Radeon RX 7800 XT (AMD), цена: 48000,00 руб., видеопамять: 16 ГБ
Итого: 53000,00 руб.


### 9.6. Статический класс `ComputerFactory`

In [14]:
Console.WriteLine("--- Офисный компьютер ---");
var officePc = ComputerFactory.CreateOfficeComputer();
officePc.ShowConfiguration();

Console.WriteLine();
Console.WriteLine("--- Игровой компьютер ---");
var gamingPc = ComputerFactory.CreateGamingComputer();
gamingPc.ShowConfiguration();

Console.WriteLine();
Console.WriteLine("--- Generic-метод CreateCustomComputer ---");
var customPc = ComputerFactory.CreateCustomComputer(
    "Сборка на заказ",
    new CPU("Ryzen 9 7950X", "AMD", 55000m, 16, 4.5),
    new RAM("Trident Z5 64GB", "G.Skill", 18000m, 64, "DDR5"),
    new GPU("GeForce RTX 4090", "NVIDIA", 160000m, 24),
    new HDD("IronWolf 4TB", "Seagate", 9000m, 4000, 5900));
customPc.ShowConfiguration();

--- Офисный компьютер ---
[Офисный ПК] Процессор Core i3-12100 установлен в сокет материнской платы.
[Офисный ПК] Модуль памяти Vengeance 8GB установлен в слот DIMM.
[Офисный ПК] Жёсткий диск Barracuda 1TB подключён к интерфейсу SATA.
Конфигурация компьютера "Офисный ПК":
  - CPU: Core i3-12100 (Intel), цена: 11000,00 руб., ядер: 4, частота: 3,3 ГГц
  - RAM: Vengeance 8GB (Corsair), цена: 2500,00 руб., объём: 8 ГБ, тип: DDR4
  - HDD: Barracuda 1TB (Seagate), цена: 3500,00 руб., объём: 1000 ГБ, скорость: 7200 об/мин
Итого: 17000,00 руб.

--- Игровой компьютер ---
[Игровой ПК] Процессор Ryzen 7 7800X3D установлен в сокет материнской платы.
[Игровой ПК] Модуль памяти Fury Beast 32GB установлен в слот DIMM.
[Игровой ПК] Видеокарта GeForce RTX 4070 установлена в слот PCI-Express.
[Игровой ПК] Жёсткий диск Barracuda 2TB подключён к интерфейсу SATA.
Конфигурация компьютера "Игровой ПК":
  - CPU: Ryzen 7 7800X3D (AMD), цена: 38000,00 руб., ядер: 8, частота: 4,2 ГГц
  - RAM: Fury Beast 32GB (Ki

## Вывод

В ходе лабораторной работы была создана система классов «Компьютер» и его компонентов:

1. **Абстрактный класс `Component`** задаёт общие свойства и виртуальный метод `GetInfo()`; метод `ToString()` переопределён.
2. **Наследование и перекрытие методов:** `CPU`, `RAM`, `GPU`, `HDD` наследуют `Component` и перекрывают `GetInfo()` через `override`, добавляя собственные данные.
3. **Интерфейс `IInstallable`** с методом `Install()` реализован во всех компонентах.
4. **Перегрузка операторов:** в каждом компоненте перегружены `==` и `!=` (вместе с `Equals` и `GetHashCode`).
5. **Перегрузка методов:** `GetTotalPrice()` и `GetTotalPrice(decimal)` - одно имя, разные параметры.
6. **Полиморфизм:** список `List<Component>` хранит объекты разных классов, а вызов `GetInfo()` выполняет версию метода реального типа.
7. **Generic-типы:** `AddComponent<T>` с ограничением `where T : Component, IInstallable` принимает только устанавливаемые компоненты, `RemoveComponent<T>` и `GetComponents<T>` работают с любыми компонентами, `ComputerFactory.CreateCustomComputer` собирает компьютер из компонентов заданных типов.
8. **Статический класс `ComputerFactory`** создаёт готовые конфигурации.

Generic позволяют писать один универсальный метод для разных типов с проверкой типов на этапе компиляции, а полиморфизм и перекрытие - единообразно работать с разными компонентами через общий базовый класс.